In [ ]:
import sys, os
import torch
from pymilvus import MilvusClient
from PIL import Image

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from model import GeoModel, device

# Connect to Docker Milvus (same as dataset.ipynb and lib.rs)
client = MilvusClient(uri="http://localhost:19530", token="root:Milvus", db_name="geoguessr")

# Load model
model = GeoModel().to(device)
WEIGHTS_PATH = os.environ.get("BRIDGE_WEIGHTS", "../alphabridge.pt")
if os.path.exists(WEIGHTS_PATH):
    model.bridge.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model.eval()
print(f"Connected to Milvus | Device: {device} | Collection: world_locations")

In [ ]:
def geolocate(image_path, top_k=5):
    """Image path -> top-k nearest locations from world_locations collection."""
    img = Image.open(image_path).convert("RGB")

    # 1. Image -> StreetCLIP 768d -> AlphaBridge 64d
    clip_vec = model.encode_images([img])
    with torch.no_grad():
        alpha_vec = model(clip_vec)
    query = alpha_vec.squeeze(0).cpu().tolist()

    # 2. Search Milvus (alphaearth_vec field, world_locations collection)
    results = client.search(
        collection_name="world_locations",
        data=[query],
        anns_field="alphaearth_vec",
        limit=top_k,
        output_fields=["gps", "s2sphere_boundary"]
    )

    # 3. Parse results — gps is a JSON field with lat/lon
    predictions = []
    for hit in results[0]:
        gps = hit["entity"]["gps"]
        predictions.append({
            "lat": gps["lat"],
            "lon": gps["lon"],
            "s2": hit["entity"]["s2sphere_boundary"],
            "distance": hit["distance"]
        })

    return predictions

# Quick test
# predictions = geolocate("path/to/test/image.jpg")
# for p in predictions:
#     print(f"  ({p['lat']:.4f}, {p['lon']:.4f}) s2={p['s2']} dist={p['distance']:.4f}")

Interaction notebook — queries the `world_locations` collection in the Docker Milvus instance (`geoguessr` db). Uses the same schema as `dataset.ipynb` and `inference/src/lib.rs`.